## **Importing the necessary Libraries.**

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix
from sklearn.tree import DecisionTreeRegressor, DecisionTreeClassifier
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.neighbors import KNeighborsRegressor, KNeighborsClassifier
from sklearn.model_selection import GridSearchCV
import warnings
warnings.filterwarnings('ignore')

**1.1.** Reading in the data.

In [2]:
cis_ml= pd.read_csv("./cis_data_cleaned_for_ml.csv")
cis_ml.head()

,year,geography,weight,gender,education,immigrant_status,earnings,wages_salary,total_income,region
0,2018,Ontario,208.9708,Female,Postsecondary certificate or diploma,Born in Canada (non-immigrant),50000,50000,50125,Ontario
1,2018,Ontario,208.9708,Male,University degree,Born in Canada (non-immigrant),49000,49000,49225,Ontario
2,2018,British Columbia,1101.2217,Female,University degree,Born in Canada (non-immigrant),92500,92500,93900,Rest of Canada
3,2018,British Columbia,1101.2217,Female,University degree,Born in Canada (non-immigrant),49000,0,49375,Rest of Canada
4,2018,Saskatchewan,165.3016,Female,High school diploma,Born in Canada (non-immigrant),21000,0,21000,Rest of Canada


**1.2.** Creating a copy of the data.

In [3]:
df = cis_ml.copy()

In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 346897 entries, 0 to 346896
Data columns (total 10 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   year              346897 non-null  int64  
 1   geography         346897 non-null  object 
 2   weight            346897 non-null  float64
 3   gender            346897 non-null  object 
 4   education         346897 non-null  object 
 5   immigrant_status  346897 non-null  object 
 6   earnings          346897 non-null  int64  
 7   wages_salary      346897 non-null  int64  
 8   total_income      346897 non-null  int64  
 9   region            346897 non-null  object 
dtypes: float64(1), int64(4), object(5)
memory usage: 26.5+ MB


**1.3.** Creating a new column called "Income Level" from the "total income" column

**1.3.1.** Removing invalid income values.

In [5]:
df= df[df['total_income'].notna()]
df= df[df['total_income'] > 0]
df.head()

,year,geography,weight,gender,education,immigrant_status,earnings,wages_salary,total_income,region
0,2018,Ontario,208.9708,Female,Postsecondary certificate or diploma,Born in Canada (non-immigrant),50000,50000,50125,Ontario
1,2018,Ontario,208.9708,Male,University degree,Born in Canada (non-immigrant),49000,49000,49225,Ontario
2,2018,British Columbia,1101.2217,Female,University degree,Born in Canada (non-immigrant),92500,92500,93900,Rest of Canada
3,2018,British Columbia,1101.2217,Female,University degree,Born in Canada (non-immigrant),49000,0,49375,Rest of Canada
4,2018,Saskatchewan,165.3016,Female,High school diploma,Born in Canada (non-immigrant),21000,0,21000,Rest of Canada


**1.3.2.** Splitting the training and testing dataset by Region.

In [6]:
train_df = df[df['region']=="Ontario"].copy()
test_df = df[df['region'] == "Rest of Canada"].copy()

**1.3.3.** Creating the Ontario-based income cutoffs.

**1.3.3.1.** Creating the tertile cutoffs.

In [7]:
quantile_1 = train_df['total_income'].quantile(1/3) #Low Boundary
quantile_2 = train_df['total_income'].quantile(2/3) #Medium Boundary

**1.3.3.2.** Using the Ontario cutoffs to label both the training and testing dataset.

This is because we are using Ontario as a reference scale to evaluate whether Ontario-based income structures generalizes with to the rest of Canada.

In [8]:
def label_income(x):
  if x <= quantile_1:
    return 'Low'
  elif quantile_1 < x <= quantile_2:
    return 'Medium'
  else:
    return 'High'

In [9]:
train_df['income_level'] = train_df['total_income'].apply(label_income)
test_df['income_level'] = test_df['total_income'].apply(label_income)

In [10]:
print(train_df['income_level'].value_counts(dropna= False))
print()
print(test_df['income_level'].value_counts(dropna= False))

income_level
Low       34614
Medium    34609
High      34592
Name: count, dtype: int64

income_level
Medium    85751
Low       80848
High      71302
Name: count, dtype: int64


In [11]:
print(train_df.info())
print()
print(test_df.info())

<class 'pandas.core.frame.DataFrame'>
Index: 103815 entries, 0 to 346896
Data columns (total 11 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   year              103815 non-null  int64  
 1   geography         103815 non-null  object 
 2   weight            103815 non-null  float64
 3   gender            103815 non-null  object 
 4   education         103815 non-null  object 
 5   immigrant_status  103815 non-null  object 
 6   earnings          103815 non-null  int64  
 7   wages_salary      103815 non-null  int64  
 8   total_income      103815 non-null  int64  
 9   region            103815 non-null  object 
 10  income_level      103815 non-null  object 
dtypes: float64(1), int64(4), object(6)
memory usage: 9.5+ MB
None

<class 'pandas.core.frame.DataFrame'>
Index: 237901 entries, 2 to 346894
Data columns (total 11 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  


# **2.0.** Machine Learning Preprocessing.

##**2.1.** Defining the features and Targets variable.

In [12]:
X = train_df.drop(columns=['total_income','income_level', 'geography', 'region'])
y_reg = train_df['total_income']
y_clf = train_df['income_level']

In [13]:
X.columns

Index(['year', 'weight', 'gender', 'education', 'immigrant_status', 'earnings',
       'wages_salary'],
      dtype='object')

In [14]:
numeric_features = ["year", "weight", "earnings", "wages_salary"]
categorical_features = ["gender", "education", "immigrant_status"]

## **2.2.** Splitting the training data into training and validation data.

### **2.2.1.** **For Regression.**

In [15]:
X_train_reg, X_val_reg, y_train_reg, y_val_reg = train_test_split(X, y_reg, test_size=0.2, random_state=42)

### **2.2.2.** **For Classification.**

In [16]:
X_train_clf, X_val_clf, y_train_clf, y_val_clf = train_test_split(X, y_clf, test_size=0.2, random_state=42)

## **2.3.** Building the preprocessor for encoding our categorical values and scaling our Numerical values.

- The **Pipeline** is used to combine preprocessing steps (encoding and scaling) and model training into a single object, instead of writing them as seperate code blocks.

- This helps to ensure that preprocessing is learned only from the training data and then applied to the validation and test data which helps prevent data leakage.

- The **Column Transformer** is used to apply different preprocessing methods to different groups of columns at the same time.

- This makes preprocessing step cleaner, faster and easier to manage within the machine learning pipeline.

### **2.3.1.** For the numeric values we Scale.
- Scaling is applied to numeric features to ensure that all values are on a comparable range, preventing the model from giving too much importance to large numbers just because they are big.

In [17]:
numeric_transformer = Pipeline(steps=[
    ('scaler', MinMaxScaler())
])

### **2.3.2.** For the Categorical values we Encode.
- One-Hot encoding is used to convert categories into Numerical format. Althrough there's a real world progression in Education, One-Hot encoding is applied to avoid forcing any false rank, thereby allowing the model to learn category effects directly from the data.

In [18]:
categorical_transformer = Pipeline(steps=[
    ('onehot', OneHotEncoder(drop='first', handle_unknown='ignore'))
])

### **2.3.3.** Combining both the numeric and categorical transformer together using the preprocessor.

In [19]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ]
)

# **3.0.** Models Building.

#### Based on the requirement for this aspect of the project, we would be making use of all the models we've been thought in class for both Regression and Classification.


##### As this would also help to validate the reason why we are picking our model.

**Column transfer** - Helps to apply different preprocessing to different sets of columns at the same time.

**Pipeline:** - Cleans the data using the Preprocessor, Then proceeds to train the model.

## **3.1.** Regression Models.

For the regression model we evaluate performance using R², Mean Absolute Error(MAE), and Root Mean Squared Error (RMSE).

- **R²**: Tells us how well the model explains income differences.

- **MAE**: Tells us how wrong the model is on average.

- **RMSE**: Tells us if the model sometimes makes very big mistakes.

### **3.1.1.** Linear Regression - The baseline Regression Model.

In [20]:
linreg_pipe = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', LinearRegression())
])

#### **3.1.1.1.** **Training** the Linear Regression Model

In [21]:
linreg_pipe.fit(X_train_reg, y_train_reg)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('scaler',
                                                                   MinMaxScaler())]),
                                                  ['year', 'weight', 'earnings',
                                                   'wages_salary']),
                                                 ('cat',
                                                  Pipeline(steps=[('onehot',
                                                                   OneHotEncoder(drop='first',
                                                                                 handle_unknown='ignore'))]),
                                                  ['gender', 'education',
                                                   'immigrant_status'])])),
                ('model', LinearRegression())])

#### **3.1.1.2.** **Validating** the Linear Regression Model.

In [22]:
val_pred = linreg_pipe.predict(X_val_reg)

In [23]:
print(f"R2: {r2_score(y_val_reg, val_pred):.4f}")
print(f"MAE: {mean_absolute_error(y_val_reg, val_pred):.4f}")
print(f"RMSE: {np.sqrt(mean_squared_error(y_val_reg, val_pred)):.4f}")

R2: 0.7115
MAE: 17867.3745
RMSE: 32810.3987


**Explanation.**

**R^2:** The linear regression model explains roughly 71% of income differences within Ontario.

**MAE:** The model's income prediction is wrong with about approximately $18k.

**RMSE:** This shows how worst the mistakes can be when the model makes mistakes.

#### **3.1.1.3.** **Testing** the Linear Regression Model.

In [24]:
X_test = test_df.drop(columns=['total_income','income_level', 'geography', 'region'])
y_test_reg = test_df['total_income']

test_pred = linreg_pipe.predict(X_test)

In [25]:
print(f"R2: {r2_score(y_test_reg, test_pred):.4f}")
print(f"MAE: {mean_absolute_error(y_test_reg, test_pred):.4f}")
print(f"RMSE: {np.sqrt(mean_squared_error(y_test_reg, test_pred)):.4f}")

R2: 0.7278
MAE: 16060.0313
RMSE: 25907.4023


**Explanation.**
- **R^2:** This shows that the way income works in Ontario is not very different from how it works in the Rest of Canada.

- **MAE:** The model makes a slightly better guess in the Rest of Canada.

- **RMSE:** This shows that the model makes fewer extreme mistakes when applied to the rest of Canada than in Ontario.


### **3.1.2.** Decision Trees Regression.

Captures non-linear patterns.

In [26]:
decisiontree_reg = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', DecisionTreeRegressor(random_state=42, max_depth=5))
])

Max depth is used to limit how the tree grows to help prevent overfitting.

#### **3.1.2.1.** **Training** the Decision Tree Regression Model.

In [27]:
decisiontree_reg.fit(X_train_reg, y_train_reg)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('scaler',
                                                                   MinMaxScaler())]),
                                                  ['year', 'weight', 'earnings',
                                                   'wages_salary']),
                                                 ('cat',
                                                  Pipeline(steps=[('onehot',
                                                                   OneHotEncoder(drop='first',
                                                                                 handle_unknown='ignore'))]),
                                                  ['gender', 'education',
                                                   'immigrant_status'])])),
                ('model', DecisionTreeRegressor(max_depth=5, random_state=42))])

#### **3.1.2.2.** **Validating** the Decision Tree Regression Model.

In [28]:
val_decisiontree_pred = decisiontree_reg.predict(X_val_reg)

val_r2_dt = r2_score(y_val_reg, val_decisiontree_pred)
val_mae_dt = mean_absolute_error(y_val_reg, val_decisiontree_pred)
val_rmse_dt = np.sqrt(mean_squared_error(y_val_reg, val_decisiontree_pred))

print(f"R2: {val_r2_dt:.4f}")
print(f"MAE: {val_mae_dt:.4f}")
print(f"RMSE: {val_rmse_dt:.4f}")

R2: 0.7470
MAE: 15093.9218
RMSE: 30723.9693


#### **3.1.2.3.** **Testing** the Decision Tree Regression Model.

In [29]:
test_pred_dt = decisiontree_reg.predict(X_test)

test_r2_dt = r2_score(y_test_reg, test_pred_dt)
test_mae_dt = mean_absolute_error(y_test_reg, test_pred_dt)
test_rmse_dt = np.sqrt(mean_squared_error(y_test_reg, test_pred_dt))

print(f"R2: {test_r2_dt:.4f}")
print(f"MAE: {test_mae_dt:.4f}")
print(f"RMSE: {test_rmse_dt:.4f}")

R2: 0.7557
MAE: 13411.3234
RMSE: 24544.0401


### **3.1.3.** Random Forest Regression.

In [30]:
randforest_reg = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', RandomForestRegressor(random_state= 42, n_estimators=200, n_jobs= -1))
])

#### **3.1.3.1.** **Training** the Random Forest Regression Model.

In [31]:
randforest_reg.fit(X_train_reg, y_train_reg)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('scaler',
                                                                   MinMaxScaler())]),
                                                  ['year', 'weight', 'earnings',
                                                   'wages_salary']),
                                                 ('cat',
                                                  Pipeline(steps=[('onehot',
                                                                   OneHotEncoder(drop='first',
                                                                                 handle_unknown='ignore'))]),
                                                  ['gender', 'education',
                                                   'immigrant_status'])])),
                ('model',
                 RandomForestRegressor(n_estimators=200, n_jobs=-1,
                                       random_state=42))])

#### **3.1.3.2.** **Validating** the Random Forest Regression Model.

In [32]:
val_regtree_pred = randforest_reg.predict(X_val_reg)

val_r2_rf = r2_score(y_val_reg, val_regtree_pred)
val_mae_rf = mean_absolute_error(y_val_reg, val_regtree_pred)
val_rmse_rf = np.sqrt(mean_squared_error(y_val_reg, val_regtree_pred))

print(f"R2: {val_r2_rf:.4f}")
print(f"MAE: {val_mae_rf:.4f}")
print(f"RMSE: {val_rmse_rf:.4f}")

R2: 0.7125
MAE: 15705.2988
RMSE: 32752.1651


#### **3.1.3.3.** **Testing** the Random Forest Regression Model.

In [33]:
test_pred_rf = randforest_reg.predict(X_test)

test_r2_rf = r2_score(y_test_reg, test_pred_rf)
test_mae_rf = mean_absolute_error(y_test_reg, test_pred_rf)
test_rmse_rf = np.sqrt(mean_squared_error(y_test_reg, test_pred_rf))

print(f"R2: {test_r2_rf:.4f}")
print(f"MAE: {test_mae_rf:.4f}")
print(f"RMSE: {test_rmse_rf:.4f}")

R2: 0.6989
MAE: 14946.4454
RMSE: 27246.0925


### **3.1.4.** K-Nearest Neighbour Regression.

In [34]:
knn_reg = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', KNeighborsRegressor(n_neighbors=5))
])

#### **3.1.4.1.** **Training** the K-Nearest Neighbour Regression Model.

In [35]:
knn_reg.fit(X_train_reg, y_train_reg)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('scaler',
                                                                   MinMaxScaler())]),
                                                  ['year', 'weight', 'earnings',
                                                   'wages_salary']),
                                                 ('cat',
                                                  Pipeline(steps=[('onehot',
                                                                   OneHotEncoder(drop='first',
                                                                                 handle_unknown='ignore'))]),
                                                  ['gender', 'education',
                                                   'immigrant_status'])])),
                ('model', KNeighborsRegressor())])

#### **3.1.4.2.** **Validating** the K-Nearest Neighbour Regression Model.

In [36]:
val_knn_reg = knn_reg.predict(X_val_reg)

val_r2_knn = r2_score(y_val_reg, val_knn_reg)
val_mae_knn = mean_absolute_error(y_val_reg, val_knn_reg)
val_rmse_knn = np.sqrt(mean_squared_error(y_val_reg, val_knn_reg))

print(f"R2: {val_r2_knn:.4f}")
print(f"MAE: {val_mae_knn:.4f}")
print(f"RMSE: {val_rmse_knn:.4f}")

R2: 0.6966
MAE: 16489.5877
RMSE: 33644.9737


#### **3.1.4.3.** **Testing** the K-Nearest Neighbour Regression Model.

In [37]:
test_pred_knn = knn_reg.predict(X_test)

test_r2_knn = r2_score(y_test_reg, test_pred_knn)
test_mae_knn = mean_absolute_error(y_test_reg, test_pred_knn)
test_rmse_knn = np.sqrt(mean_squared_error(y_test_reg, test_pred_knn))

print(f"R2: {test_r2_knn:.4f}")
print(f"MAE: {test_mae_knn:.4f}")
print(f"RMSE: {test_rmse_knn:.4f}")

R2: 0.6916
MAE: 15233.7045
RMSE: 27576.0553


## **3.2.** Classification Models.

For the Classification model, we evaluate performance using Accuracy, Precision, F1-score and the Confusion Matrix.

- **Accuracy:** Tells us how often the model gets it right overall.

- **Precision**: Tells us how often the model guesses correctly, when it predicts a particular income group.

- **Recall**: Tells us how many actual cases in each income group the model is able to correctly identify.

- **F1-Score**: Provides a balance between Precision and Recall. This is useful because income groups can still overlap and may not be equally easy to predict, even after grouping incomes using quantiles.

- **Confusion Matrix:** Displays how predictions are distributed across income groups and helps identify where the model confuses one income level for another.

- **Classification Report:** this summaries all our metrics and how they support each income level in one table.

### **3.2.1.** Logistic Regression.

In [38]:
logreg_pipe = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', LogisticRegression(max_iter=2000, class_weight= 'balanced'))])

#### **3.2.1.1.** **Training** the Logistic Regression Model.

In [39]:
logreg_pipe.fit(X_train_clf, y_train_clf)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('scaler',
                                                                   MinMaxScaler())]),
                                                  ['year', 'weight', 'earnings',
                                                   'wages_salary']),
                                                 ('cat',
                                                  Pipeline(steps=[('onehot',
                                                                   OneHotEncoder(drop='first',
                                                                                 handle_unknown='ignore'))]),
                                                  ['gender', 'education',
                                                   'immigrant_status'])])),
                ('model',
                 LogisticRegression(class_weight='balanced', max_iter=2000))])

#### **3.2.1.2.** **Validating** the Logistic Regression Model.

In [40]:
logval_pred = logreg_pipe.predict(X_val_clf)

In [41]:
print(f"Accuracy: {accuracy_score(y_val_clf, logval_pred):.4f}")
print()
print(f"Precision: {precision_score(y_val_clf, logval_pred, average= 'weighted'):.4f}")
print()
print(f"Recall: {recall_score(y_val_clf, logval_pred, average= 'weighted'):.4f}")
print()
print(f"F1-Score: {f1_score(y_val_clf, logval_pred, average= 'weighted'):.4f}")
print()
print("Confusion Matrix:")
print(confusion_matrix(y_val_clf, logval_pred))
print()
print("Classification Report:")
print(classification_report(y_val_clf, logval_pred))

Accuracy: 0.6816

Precision: 0.6841

Recall: 0.6816

F1-Score: 0.6745

Confusion Matrix:
[[5323  890  767]
 [  16 5602 1201]
 [1288 2449 3227]]

Classification Report:
              precision    recall  f1-score   support

        High       0.80      0.76      0.78      6980
         Low       0.63      0.82      0.71      6819
      Medium       0.62      0.46      0.53      6964

    accuracy                           0.68     20763
   macro avg       0.68      0.68      0.67     20763
weighted avg       0.68      0.68      0.67     20763



**Explanation:**
- **Accuracy:** This means that out of every 10 people, the model correctly places 7 people in their right income group.
- **Precision:** This shows that when the model predicts high income, it is usually right compared to the Medium and Low income.
- **Recall:** The model better identifies Low and High income groups more reliably than the Medium group.
- **F1-Score:** This model is reliable for Low and High income groups, but weakest is the hardest to classify.

**The model could be struggling with medium income group becaue rget could be an overlap sometimes on the low and high income groups.**

#### **3.2.1.3.** **Testing** the Logistic Regression Model.

In [42]:
X_test_clf = test_df.drop(columns=["total_income", "income_level", "geography"], errors="ignore")
y_test_clf = test_df["income_level"]

In [43]:
logtest_pred = logreg_pipe.predict(X_test_clf)

In [44]:
print(f"Accuracy: {accuracy_score(y_test_clf, logtest_pred):.4f}")
print()
print(f"Precision: {precision_score(y_test_clf, logtest_pred, average= 'weighted'):.4f}")
print()
print(f"Recall: {recall_score(y_test_clf, logtest_pred, average= 'weighted'):.4f}")
print()
print(f"F1-Score: {f1_score(y_test_clf, logtest_pred, average= 'weighted'):.4f}")
print()
print("Confusion Matrix:")
print(confusion_matrix(y_test_clf, logtest_pred))
print()
print("Classification Report:")
print(classification_report(y_test_clf, logtest_pred))

Accuracy: 0.6887

Precision: 0.6867

Recall: 0.6887

F1-Score: 0.6834

Confusion Matrix:
[[56243  6478  8581]
 [  255 63692 16901]
 [15238 26610 43903]]

Classification Report:
              precision    recall  f1-score   support

        High       0.78      0.79      0.79     71302
         Low       0.66      0.79      0.72     80848
      Medium       0.63      0.51      0.57     85751

    accuracy                           0.69    237901
   macro avg       0.69      0.70      0.69    237901
weighted avg       0.69      0.69      0.68    237901



### **3.2.2.** Decision Trees Classification.

In [45]:
dt_clf_pipe = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', DecisionTreeClassifier(random_state=42, max_depth=5))
])

#### **3.2.2.1.** **Training** the Decision Tree Classification Model.

In [46]:
dt_clf_pipe.fit(X_train_clf, y_train_clf)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('scaler',
                                                                   MinMaxScaler())]),
                                                  ['year', 'weight', 'earnings',
                                                   'wages_salary']),
                                                 ('cat',
                                                  Pipeline(steps=[('onehot',
                                                                   OneHotEncoder(drop='first',
                                                                                 handle_unknown='ignore'))]),
                                                  ['gender', 'education',
                                                   'immigrant_status'])])),
                ('model',
                 DecisionTreeClassifier(max_depth=5, random_state=42))])

#### **3.2.2.2.** **Validating** the Decision Tree Classification Model.

In [47]:
val_dt_clf = dt_clf_pipe.predict(X_val_clf)

In [48]:
print(f"Accuracy: {accuracy_score(y_val_clf, val_dt_clf):.4f}")
print()
print(f"Precision: {precision_score(y_val_clf, val_dt_clf, average= 'weighted'):.4f}")
print()
print(f"Recall: {recall_score(y_val_clf, val_dt_clf, average= 'weighted'):.4f}")
print()
print(f"F1-Score: {f1_score(y_val_clf, val_dt_clf, average= 'weighted'):.4f}")
print()
print("Confusion Matrix:")
print(confusion_matrix(y_val_clf, val_dt_clf))
print()
print("Classification Report:\n", classification_report(y_val_clf, val_dt_clf))

Accuracy: 0.7405

Precision: 0.7730

Recall: 0.7405

F1-Score: 0.7353

Confusion Matrix:
[[5517  835  628]
 [ 403 6251  165]
 [ 392 2966 3606]]

Classification Report:
               precision    recall  f1-score   support

        High       0.87      0.79      0.83      6980
         Low       0.62      0.92      0.74      6819
      Medium       0.82      0.52      0.63      6964

    accuracy                           0.74     20763
   macro avg       0.77      0.74      0.74     20763
weighted avg       0.77      0.74      0.74     20763



#### **3.2.2.3.** **Testing** the Decision Tree Classification Model.

In [49]:
test_dt_clf = dt_clf_pipe.predict(X_test_clf)

In [50]:
print(f"Accuracy: {accuracy_score(y_test_clf, test_dt_clf):.4f}")
print()
print(f'Precision: {precision_score(y_test_clf, test_dt_clf, average= 'weighted'):4f}')
print()
print(f'Recall: {recall_score(y_test_clf, test_dt_clf, average= 'weighted'):4f}')
print()
print(f'F1-Score: {f1_score(y_test_clf, test_dt_clf, average= 'weighted'):4f}')
print()
print("Confusion Matrix:")
print(confusion_matrix(y_test_clf, test_dt_clf))
print()
print("Classification Report:")
print(classification_report(y_test_clf, test_dt_clf))

Accuracy: 0.7361

Precision: 0.765799

Recall: 0.736092

F1-Score: 0.728072

Confusion Matrix:
[[56627  7300  7375]
 [ 4591 74703  1554]
 [ 6685 35279 43787]]

Classification Report:
              precision    recall  f1-score   support

        High       0.83      0.79      0.81     71302
         Low       0.64      0.92      0.75     80848
      Medium       0.83      0.51      0.63     85751

    accuracy                           0.74    237901
   macro avg       0.77      0.74      0.73    237901
weighted avg       0.77      0.74      0.73    237901



### **3.2.3.** Random Forest Classification.

In [51]:
rf_clf_pipe = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', RandomForestClassifier(random_state=42, n_estimators=200, n_jobs=-1))
])

#### **3.2.3.1.** **Training** the Random Forest Classification Model.

In [52]:
rf_clf_pipe.fit(X_train_clf, y_train_clf)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('scaler',
                                                                   MinMaxScaler())]),
                                                  ['year', 'weight', 'earnings',
                                                   'wages_salary']),
                                                 ('cat',
                                                  Pipeline(steps=[('onehot',
                                                                   OneHotEncoder(drop='first',
                                                                                 handle_unknown='ignore'))]),
                                                  ['gender', 'education',
                                                   'immigrant_status'])])),
                ('model',
                 RandomForestClassifier(n_estimators=200, n_jobs=-1,
                                        random_state=42))])

#### **3.2.3.2.** **Validating** the Random Forest Classification Model.

In [53]:
val_rf_clf = rf_clf_pipe.predict(X_val_clf)

In [54]:
print(f"Accuracy: {accuracy_score(y_val_clf, val_rf_clf):.4f}")
print()
print(f"Precision: {precision_score(y_val_clf, val_rf_clf, average= 'weighted'):.4f}")
print()
print(f"Recall: {recall_score(y_val_clf, val_rf_clf, average= 'weighted'):.4f}")
print()
print(f"F1-Score: {f1_score(y_val_clf, val_rf_clf, average= 'weighted'):.4f}")
print()
print("Confusion Matrix:")
print(confusion_matrix(y_val_clf, val_rf_clf))
print()
print("Classification Report:")
print(classification_report(y_val_clf, val_rf_clf))

Accuracy: 0.7135

Precision: 0.7181

Recall: 0.7135

F1-Score: 0.7150

Confusion Matrix:
[[5406  577  997]
 [ 434 4894 1491]
 [ 592 1858 4514]]

Classification Report:
              precision    recall  f1-score   support

        High       0.84      0.77      0.81      6980
         Low       0.67      0.72      0.69      6819
      Medium       0.64      0.65      0.65      6964

    accuracy                           0.71     20763
   macro avg       0.72      0.71      0.71     20763
weighted avg       0.72      0.71      0.72     20763



#### **3.2.3.3.** **Testing** the Random Forest Classification Model.

In [55]:
test_rf_clf = rf_clf_pipe.predict(X_test_clf)

In [56]:
print(f"Accuracy: {accuracy_score(y_test_clf, test_rf_clf):.4f}")
print()
print(f"Precision: {precision_score(y_test_clf, test_rf_clf, average= 'weighted'):.4f}")
print()
print(f"Recall: {recall_score(y_test_clf, test_rf_clf, average= 'weighted'):.4f}")
print()
print(f"F1-Score: {f1_score(y_test_clf, test_rf_clf, average= 'weighted'):.4f}")
print()
print("Confusion Matrix:")
print(confusion_matrix(y_test_clf, test_rf_clf))
print()
print("Classification Report:")
print(classification_report(y_test_clf, test_rf_clf))

Accuracy: 0.6954

Precision: 0.6966

Recall: 0.6954

F1-Score: 0.6959

Confusion Matrix:
[[55894  5411  9997]
 [ 5471 55258 20119]
 [ 8034 23421 54296]]

Classification Report:
              precision    recall  f1-score   support

        High       0.81      0.78      0.79     71302
         Low       0.66      0.68      0.67     80848
      Medium       0.64      0.63      0.64     85751

    accuracy                           0.70    237901
   macro avg       0.70      0.70      0.70    237901
weighted avg       0.70      0.70      0.70    237901



### **3.2.4.** K-Nearest Neighbours Classification.

In [57]:
knn_clf_pipe = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', KNeighborsClassifier(n_neighbors=5))
])

#### **3.2.4.1.** **Training** the K-Nearest Neighbour Classification Model.

In [58]:
knn_clf_pipe.fit(X_train_clf, y_train_clf)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('scaler',
                                                                   MinMaxScaler())]),
                                                  ['year', 'weight', 'earnings',
                                                   'wages_salary']),
                                                 ('cat',
                                                  Pipeline(steps=[('onehot',
                                                                   OneHotEncoder(drop='first',
                                                                                 handle_unknown='ignore'))]),
                                                  ['gender', 'education',
                                                   'immigrant_status'])])),
                ('model', KNeighborsClassifier())])

#### **3.2.4.2.** **Validating** the K-Nearest Neighbour Classification Model.

In [59]:
val_knn_clf = knn_clf_pipe.predict(X_val_clf)

In [60]:
print(f"Accuracy: {accuracy_score(y_val_clf, val_knn_clf):.4f}")
print()
print(f"Precision: {precision_score(y_val_clf, val_knn_clf, average= 'weighted'):.4f}")
print()
print(f"Recall: {recall_score(y_val_clf, val_knn_clf, average= 'weighted'):.4f}")
print()
print(f"F1-Score: {f1_score(y_val_clf, val_knn_clf, average= 'weighted'):.4f}")
print()
print("Confusion Matrix:")
print(confusion_matrix(y_val_clf, val_knn_clf))
print()
print("Classification Report:")
print(classification_report(y_val_clf, val_knn_clf))

Accuracy: 0.7057

Precision: 0.7115

Recall: 0.7057

F1-Score: 0.7053

Confusion Matrix:
[[5379  696  905]
 [ 471 5225 1123]
 [ 634 2282 4048]]

Classification Report:
              precision    recall  f1-score   support

        High       0.83      0.77      0.80      6980
         Low       0.64      0.77      0.70      6819
      Medium       0.67      0.58      0.62      6964

    accuracy                           0.71     20763
   macro avg       0.71      0.71      0.71     20763
weighted avg       0.71      0.71      0.71     20763



#### **3.2.4.3.** **Testing** the K-Nearest Neighbour Classification Model.

In [61]:
test_knn_clf = knn_clf_pipe.predict(X_test_clf)

In [62]:
print(f"Accuracy: {accuracy_score(y_test_clf, test_knn_clf):.4f}")
print()
print(f"Precision: {precision_score(y_test_clf, test_knn_clf, average= 'weighted'):.4f}")
print()
print(f"Recall: {recall_score(y_test_clf, test_knn_clf, average= 'weighted'):.4f}")
print()
print(f"F1-Score: {f1_score(y_test_clf, test_knn_clf, average= 'weighted'):.4f}")
print()
print("Confusion Matrix:")
print(confusion_matrix(y_test_clf, test_knn_clf))
print()
print("Classification Report:")
print(classification_report(y_test_clf, test_knn_clf))

Accuracy: 0.6994

Precision: 0.7011

Recall: 0.6994

F1-Score: 0.6983

Confusion Matrix:
[[55719  5906  9677]
 [ 5819 59947 15082]
 [ 8582 26451 50718]]

Classification Report:
              precision    recall  f1-score   support

        High       0.79      0.78      0.79     71302
         Low       0.65      0.74      0.69     80848
      Medium       0.67      0.59      0.63     85751

    accuracy                           0.70    237901
   macro avg       0.71      0.70      0.70    237901
weighted avg       0.70      0.70      0.70    237901



# **4.0.** **Model Comparison Tables**

### **4.1.** **Model Comparison Table For Regression.**

### Regression Model Comparison (Target: Total Income)

| Model              | R² (Validation – Ontario) | RMSE (Validation – Ontario) | R² (Test – Rest of Canada) | RMSE (Test – Rest of Canada) |
|--------------------|---------------------------|------------------------------|-----------------------------|-------------------------------|
| Linear Regression  |               0.7115              |           32810.3987                   |                        0.7278     |                        25907.4023       |
| Decision Tree      |                    0.7470       |                   30723.9693           |                    0.7557         |                           24544.0401    |
| Random Forest      |               0.7125            |                           32752.1651   |                         0.6989    |                             27246.0925  |
| KNN Regression     |                 0.6966          |                       33644.9737       |                      0.6916       |                            27576.0553  |


The Regression Models were trained using Ontario data and tested on the Rest of Canada.
All models performed similarly across regions, this shows that the patterns learned in Ontario also apply to the rest of the country.

The Decision tree model performed the best, with the highest R² and the lowest error values.

Linear Regression also performed the well and was very stable, making it a good baseline mode.
Random Forest produced reasonable results but did not outperform the Decision Trees.

KNN Regressor performed the worst, this could be as a result of distance-based assumptions.

### **4.2.** **Model Comparison Table For Classification.**

### Classification Model Comparison (Target: Income Level)

| Model                 | Accuracy (Validation – Ontario) | F1-score (Validation – Ontario) | Accuracy (Test – Rest of Canada) | F1-score (Test – Rest of Canada) |
|-----------------------|----------------------------------|----------------------------------|-----------------------------------|-----------------------------------|
| Logistic Regression   |              0.6816                    |                       0.6745           |                    0.6887               |                               0.6834    |
| Decision Tree         |           0.7405                      |                   0.7353               |                             0.7361      |                            0.7281       |
| Random Forest         |               0.7135                   |                           0.7150       |                          0.6954         |                              0.6959     |
| KNN Classification    |                        0.7057          |                         0.7053         |                         0.6994          |                          0.6983         |


The Classification Models were used to predict income levels (Low, Medium, High) rather than exact income models.

The Decision Tree model achieved the highest accuracy and F1-Score during the Validation and testing, this shows strong and balanced performance.
Random Forest also performed well but did not outperform the Decision Tree.

Logistic Regression showed the weakest results, while KNN produced moderate performance.

Overall, the Decision Tree model was selected as the best classification model due to its accuracy, stability and ability to generalize from Ontario to the Rest of Canada.